# EDA — розвідувальний аналіз даних

Підтверджує hazard'и числами (4 креатори, repost-нулі, дрейф, брудна тривалість) і обґрунтовує вибір мітки. Записує `reports/eda_report.md` + графіки у `reports/plots/`.

> ▶️ Запускай з кореня репозиторію.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # графіки зберігаються у файли

In [ ]:
"""Exploratory Data Analysis for the ShouldIPost? TikTok dataset.

Run:  python -m notebooks.eda   (or)  python notebooks/eda.py
Produces:
  - reports/eda_report.md      (human-readable findings)
  - reports/plots/*.png        (a handful of diagnostic plots)

Goal: CONFIRM (not assume) the dataset hazards flagged in the brief:
  1. tiny data dominated by a few mega-creators
  2. repost_count is all zeros -> drop
  3. location_created is mostly null -> drop
  4. create_time spans multiple years -> drift
and gather the numbers we need to freeze a defensible "success" label.
"""

import io
import os
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw" / "train.csv"
REPORTS = ROOT / "reports"
PLOTS = REPORTS / "plots"
PLOTS.mkdir(parents=True, exist_ok=True)

POST_HOC = [
    "play_count",
    "digg_count",
    "share_count",
    "comment_count",
    "collect_count",
    "repost_count",
]

lines: list[str] = []


def out(msg: str = "") -> None:
    print(msg)
    lines.append(msg)


def section(title: str) -> None:
    out("")
    out(f"## {title}")
    out("")


def df_table(df: pd.DataFrame, floatfmt: str = "{:.3f}") -> str:
    """Render a small dataframe as a GitHub markdown table."""
    cols = list(df.columns)
    head = "| " + " | ".join(str(c) for c in cols) + " |"
    sep = "| " + " | ".join("---" for _ in cols) + " |"
    rows = [head, sep]
    for _, r in df.iterrows():
        cells = []
        for c in cols:
            v = r[c]
            if isinstance(v, float):
                cells.append(floatfmt.format(v))
            else:
                cells.append(str(v))
        rows.append("| " + " | ".join(cells) + " |")
    return "\n".join(rows)


def main() -> None:
    df = pd.read_csv(RAW)

    out("# EDA — ShouldIPost? (datahiveai/Tiktok-Videos)")
    out("")
    out("_Auto-generated by `notebooks/eda.py`. Numbers below are cited verbatim in PLAN.md / README._")

    # ---- 1. Shape & schema ------------------------------------------------
    section("1. Shape & schema")
    out(f"- Rows: **{len(df)}**, Columns: **{df.shape[1]}**")
    buf = io.StringIO()
    df.info(buf=buf)
    out("```")
    out(buf.getvalue().rstrip())
    out("```")

    # ---- 2. Missingness ---------------------------------------------------
    section("2. Missingness")
    miss = df.isna().mean().sort_values(ascending=False)
    miss_df = pd.DataFrame({"column": miss.index, "null_fraction": miss.values})
    out(df_table(miss_df))

    # ---- 3. repost_count hazard ------------------------------------------
    section("3. Hazard: repost_count")
    rc = df["repost_count"]
    out(f"- unique values: {sorted(rc.dropna().unique())[:10]}")
    out(f"- all zero? **{(rc.fillna(0) == 0).all()}** "
        f"(nonzero count = {(rc.fillna(0) != 0).sum()})")

    # ---- 4. location_created hazard --------------------------------------
    section("4. Hazard: location_created")
    lc = df["location_created"]
    out(f"- null fraction: **{lc.isna().mean():.3f}**  "
        f"(brief claimed 'mostly null' — NOT true for this snapshot)")
    out(f"- distinct values: {lc.nunique()}; examples: {lc.dropna().unique()[:8].tolist()}")
    # confound check: is location basically determined by the creator?
    per_creator_locs = df.groupby("author_unique_id")["location_created"].nunique()
    out(f"- distinct locations *within* each creator: {per_creator_locs.to_dict()}")
    out("  -> if ~1 per creator, location_created is just another creator/fame proxy.")

    # ---- 5. Creator concentration ----------------------------------------
    section("5. Hazard: creator concentration (fame skew)")
    n_creators = df["author_unique_id"].nunique()
    out(f"- distinct creators (`author_unique_id`): **{n_creators}**")
    vc = df["author_unique_id"].value_counts()
    out(f"- videos/creator: min={vc.min()}, median={vc.median():.0f}, "
        f"mean={vc.mean():.1f}, max={vc.max()}")
    topn = vc.head(10)
    top_df = pd.DataFrame({
        "creator": topn.index,
        "n_videos": topn.values,
        "pct_of_rows": (topn.values / len(df) * 100),
    })
    out("")
    out("Top 10 creators by row count:")
    out(df_table(top_df))
    # share of views captured by top creators
    views_by_creator = df.groupby("author_unique_id")["play_count"].sum().sort_values(ascending=False)
    top5_view_share = views_by_creator.head(5).sum() / views_by_creator.sum()
    out("")
    out(f"- Top-5 creators capture **{top5_view_share*100:.1f}%** of all `play_count`.")
    # how many creators have enough videos to compute a within-creator median
    for k in (2, 3, 5, 10):
        n_ok = (vc >= k).sum()
        rows_ok = vc[vc >= k].sum()
        out(f"- creators with >= {k} videos: {n_ok}/{n_creators} "
            f"(covering {rows_ok}/{len(df)} = {rows_ok/len(df)*100:.0f}% of rows)")

    # ---- 6. play_count & engagement --------------------------------------
    section("6. Post-hoc metrics (LABEL-ONLY, never features)")
    desc = df[POST_HOC].describe().T[["mean", "50%", "min", "max"]]
    desc.columns = ["mean", "median", "min", "max"]
    desc = desc.reset_index().rename(columns={"index": "metric"})
    out(df_table(desc, floatfmt="{:.1f}"))
    # engagement rate = (digg+share+comment+collect)/play
    eng = (df["digg_count"] + df["share_count"] + df["comment_count"] + df["collect_count"])
    er = eng / df["play_count"].replace(0, np.nan)
    out("")
    out(f"- engagement_rate = (digg+share+comment+collect)/play: "
        f"median={er.median():.4f}, mean={er.mean():.4f}, "
        f"p90={er.quantile(.9):.4f}, n_invalid(play<=0)={int((df['play_count']<=0).sum())}")

    # ---- 7. create_time drift --------------------------------------------
    section("7. Hazard: create_time drift")
    ct = pd.to_datetime(df["create_time"], unit="s", utc=True)
    out(f"- range: **{ct.min().date()} → {ct.max().date()}** "
        f"(span ≈ {(ct.max()-ct.min()).days} days)")
    n_garbage = int((df["create_time"].fillna(0) < 1_300_000_000).sum())  # < ~2011
    out(f"- rows with implausible epoch (< 2011, i.e. epoch~0 garbage): **{n_garbage}**")
    by_year = ct.dt.year.value_counts().sort_index()
    yr_df = pd.DataFrame({"year": by_year.index, "n_videos": by_year.values})
    out(df_table(yr_df))

    # ---- 8. duration ------------------------------------------------------
    section("8. duration (pre-post feature)")
    d = df["duration"]
    out(f"- seconds: min={d.min()}, median={d.median():.0f}, mean={d.mean():.1f}, "
        f"max={d.max()}, n_zero={(d==0).sum()}, n_null={d.isna().sum()}")
    out(f"- quantiles: {d.quantile([.25,.5,.75,.9,.99]).round(1).to_dict()}")

    # ---- 9. description ---------------------------------------------------
    section("9. description / caption (pre-post feature)")
    txt = df["description"].fillna("")
    char_len = txt.str.len()
    word_len = txt.str.split().apply(len)
    n_hashtags = txt.str.count(r"#\w+")
    n_mentions = txt.str.count(r"@\w+")
    out(f"- n_empty_caption: {(char_len==0).sum()} ({(char_len==0).mean()*100:.1f}%)")
    out(f"- char_len: median={char_len.median():.0f}, mean={char_len.mean():.1f}, max={char_len.max()}")
    out(f"- word_len: median={word_len.median():.0f}, mean={word_len.mean():.1f}, max={word_len.max()}")
    out(f"- hashtags/caption: median={n_hashtags.median():.0f}, mean={n_hashtags.mean():.2f}, max={n_hashtags.max()}")
    out(f"- mentions/caption: median={n_mentions.median():.0f}, mean={n_mentions.mean():.2f}, max={n_mentions.max()}")
    out(f"- has '?' : {(txt.str.contains('?', regex=False)).mean()*100:.1f}%")

    # ---- 10. fame vs quality demonstration -------------------------------
    section("10. Why within-creator labeling matters")
    # variance of mean log-views explained by creator
    logv = np.log1p(df["play_count"])
    grand = logv.mean()
    ss_tot = float(((logv - grand) ** 2).sum())
    creator_mean = df.assign(logv=logv).groupby("author_unique_id")["logv"].transform("mean")
    ss_between = float(((creator_mean - grand) ** 2).sum())
    eta2 = ss_between / ss_tot if ss_tot else float("nan")
    out(f"- Share of variance in log(play_count) explained purely by creator identity "
        f"(eta^2): **{eta2:.2f}**.")
    out("  -> A large value means raw views mostly encode *who posted*, not *how good the video is*; ")
    out("     this is the core argument for a within-creator relative success label.")

    # ---- plots ------------------------------------------------------------
    # views distribution (log)
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(np.log10(df["play_count"].replace(0, np.nan).dropna()), bins=40, color="#FE2C55")
    ax.set_title("log10(play_count) distribution")
    ax.set_xlabel("log10(views)")
    ax.set_ylabel("videos")
    fig.tight_layout()
    fig.savefig(PLOTS / "play_count_log.png", dpi=110)
    plt.close(fig)

    # creator concentration
    fig, ax = plt.subplots(figsize=(6, 4))
    topn.iloc[::-1].plot.barh(ax=ax, color="#25F4EE")
    ax.set_title("Top creators by #videos")
    ax.set_xlabel("videos")
    fig.tight_layout()
    fig.savefig(PLOTS / "creator_concentration.png", dpi=110)
    plt.close(fig)

    # videos over time
    fig, ax = plt.subplots(figsize=(6, 4))
    ct.dt.to_period("M").value_counts().sort_index().plot(ax=ax, color="#000000")
    ax.set_title("Videos per month (create_time)")
    ax.set_ylabel("videos")
    fig.tight_layout()
    fig.savefig(PLOTS / "videos_over_time.png", dpi=110)
    plt.close(fig)

    # engagement rate distribution
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(er.dropna().clip(upper=er.quantile(0.99)), bins=40, color="#FE2C55")
    ax.set_title("Engagement rate distribution (clipped p99)")
    ax.set_xlabel("(digg+share+comment+collect)/play")
    fig.tight_layout()
    fig.savefig(PLOTS / "engagement_rate.png", dpi=110)
    plt.close(fig)

    REPORTS.mkdir(parents=True, exist_ok=True)
    (REPORTS / "eda_report.md").write_text("\n".join(lines), encoding="utf-8")
    out("")
    out(f"_Plots written to {PLOTS.relative_to(ROOT)}/_")
    print(f"\n[ok] wrote {REPORTS/'eda_report.md'} and 4 plots")

### Запуск EDA

In [ ]:
main()
print('Звіт: reports/eda_report.md; графіки: reports/plots/')